# AI-Driven Automated Colorectal Polyp Assessment
## ResUNet++ Hybrid Model for Polyp Segmentation and Evaluation

**Academic Project**: AI-Driven Automated Colorectal Polyp Assessment  
**Target Architecture**: **ResUNet++** (Residual Conv + Squeeze-and-Excitation + ASPP + Attention Gates)  
**Dataset**: Roboflow Universe — Kvasir-SEG (`teste-mhypc/kvasir-seg-gouny`)  
**Recommended Runtime**: Google Colab with GPU (Runtime > Change runtime type > T4 GPU)  

> **Clinical Disclaimer**: This application is an academic computer-vision research prototype. It is NOT a clinically certified medical device and is not intended to diagnose colorectal neoplasms or replace endoscopic evaluation by certified clinicians.

### Step 1: Verify Hardware & GPU Acceleration
Ensure PyTorch has access to an NVIDIA GPU (T4, V100, or A100).

In [ ]:
import torch

print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU detected: {gpu_name} ({gpu_mem:.2f} GB VRAM)")
    device = torch.device("cuda")
else:
    print("⚠️ WARNING: No GPU detected! Running on CPU will be slow.")
    print("   Please navigate to: Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU")
    device = torch.device("cpu")


### Step 2: Install Required Libraries

In [ ]:
# Install dependencies needed for Roboflow download, image processing, and metrics
!pip install -q roboflow pycocotools pyyaml albumentations matplotlib pillow


### Step 3: Imports & Reproducibility Setup

In [ ]:
import os
import sys
import time
import math
import random
import glob
import json
import shutil
from pathlib import Path

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Set seeds for exact reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
print("Seeds configured for reproducibility (seed=42).")


### Step 4: Download & Prepare Kvasir-SEG Dataset
You can supply your Roboflow API key via the interactive prompt or Colab `userdata`.
If you don't have an API key right now, you can also test the pipeline with synthetic demo data.

In [ ]:
# Dataset Configuration
ROBOFLOW_WORKSPACE = "teste-mhypc"
ROBOFLOW_PROJECT = "kvasir-seg-gouny"
ROBOFLOW_VERSION = 4

RAW_DATA_DIR = "data/raw"
PROCESSED_DATA_DIR = "data/processed"

# Check Colab secrets or environment
api_key = ""
try:
    from google.colab import userdata
    api_key = userdata.get("ROBOFLOW_API_KEY")
except Exception:
    pass

if not api_key:
    api_key = os.environ.get("ROBOFLOW_API_KEY", "")

if not api_key:
    print("Enter your Roboflow API Key (or leave blank to create a synthetic demonstration dataset):")
    api_key = input("Roboflow API Key: ").strip()

if api_key:
    from roboflow import Roboflow
    rf = Roboflow(api_key=api_key)
    project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
    print(f"Downloading {ROBOFLOW_PROJECT} v{ROBOFLOW_VERSION}...")
    try:
        dataset = project.version(ROBOFLOW_VERSION).download("png-mask", location=RAW_DATA_DIR)
        print("Downloaded in png-mask format!")
    except Exception as e:
        print(f"png-mask not available ({e}), trying 'coco' format...")
        dataset = project.version(ROBOFLOW_VERSION).download("coco", location=RAW_DATA_DIR)
else:
    print("No API key provided: Creating synthetic colonoscopy demo dataset for testing the pipeline...")
    for split, count in [('train', 30), ('val', 10), ('test', 10)]:
        img_dir = Path(PROCESSED_DATA_DIR) / "images" / split
        mask_dir = Path(PROCESSED_DATA_DIR) / "masks" / split
        img_dir.mkdir(parents=True, exist_ok=True)
        mask_dir.mkdir(parents=True, exist_ok=True)
        for i in range(count):
            # Create synthetic colonoscopy-like image (pinkish hue with a circular polyp)
            img = np.random.normal(loc=[160, 90, 80], scale=[25, 20, 20], size=(256, 256, 3)).clip(0, 255).astype(np.uint8)
            mask = np.zeros((256, 256), dtype=np.uint8)
            # Add synthetic polyp
            cx, cy = np.random.randint(60, 196, 2)
            r = np.random.randint(20, 50)
            yy, xx = np.ogrid[:256, :256]
            circle = ((xx - cx) ** 2 + (yy - cy) ** 2) <= r ** 2
            mask[circle] = 255
            img[circle] = np.clip(img[circle].astype(int) + 40, 0, 255).astype(np.uint8)
            Image.fromarray(img).save(img_dir / f"polyps_{i:03d}.jpg")
            Image.fromarray(mask).save(mask_dir / f"polyps_{i:03d}.png")
    print("Synthetic dataset generated in data/processed/")


### Step 5: Organize & Validate Standard Dataset Structure
Ensures images and masks are matched 1:1, binarized (0 background, 1 polyp), and split into train/val/test.

In [ ]:
def standardize_dataset(raw_dir=RAW_DATA_DIR, proc_dir=PROCESSED_DATA_DIR):
    raw = Path(raw_dir)
    proc = Path(proc_dir)
    
    if not raw.exists():
        print(f"Raw directory {raw_dir} does not exist, checking processed directory...")
        return
        
    for split in ["train", "valid", "test"]:
        s_dir = raw / split
        if not s_dir.exists():
            continue
        out_split = "val" if split == "valid" else split
        out_img_dir = proc / "images" / out_split
        out_mask_dir = proc / "masks" / out_split
        out_img_dir.mkdir(parents=True, exist_ok=True)
        out_mask_dir.mkdir(parents=True, exist_ok=True)
        
        coco_file = s_dir / "_annotations.coco.json"
        if coco_file.exists():
            # COCO JSON conversion
            print(f"Processing COCO annotations for {split}...")
            with open(coco_file, 'r') as f:
                coco = json.load(f)
            id_to_img = {im["id"]: im for im in coco["images"]}
            for im_id, im_info in id_to_img.items():
                h, w = im_info["height"], im_info["width"]
                mask = Image.new("L", (w, h), 0)
                from PIL import ImageDraw
                draw = ImageDraw.Draw(mask)
                for ann in coco.get("annotations", []):
                    if ann["image_id"] == im_id:
                        for poly in ann.get("segmentation", []):
                            if len(poly) >= 6:
                                pts = [(poly[k], poly[k+1]) for k in range(0, len(poly), 2)]
                                draw.polygon(pts, fill=255)
                base = Path(im_info["file_name"]).stem
                mask.save(out_mask_dir / f"{base}.png")
                src_img = s_dir / im_info["file_name"]
                if src_img.exists():
                    shutil.copy2(src_img, out_img_dir / f"{base}.jpg")
        else:
            # Direct image & mask copy
            for f in s_dir.glob("*.jpg"):
                shutil.copy2(f, out_img_dir / f.name)
            mask_dir_candidates = [d for d in s_dir.iterdir() if d.is_dir() and "mask" in d.name.lower()]
            if mask_dir_candidates:
                for mf in mask_dir_candidates[0].glob("*.png"):
                    shutil.copy2(mf, out_mask_dir / mf.name)

standardize_dataset()

# Check counts
for split in ["train", "val", "test"]:
    n_img = len(list((Path(PROCESSED_DATA_DIR) / "images" / split).glob("*.*")))
    n_mask = len(list((Path(PROCESSED_DATA_DIR) / "masks" / split).glob("*.*")))
    print(f"{split.upper()}: {n_img} images, {n_mask} masks")


### Step 6: PyTorch Dataset Class with Synchronized Augmentations

In [ ]:
class PolypDataset(Dataset):
    def __init__(self, images_dir, masks_dir, img_size=(256, 256), augment=False):
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir)
        self.img_size = img_size
        self.augment = augment
        
        img_files = sorted(list(self.images_dir.glob("*.jpg")) + list(self.images_dir.glob("*.png")))
        self.pairs = []
        for img_p in img_files:
            mask_candidates = [
                self.masks_dir / f"{img_p.stem}.png",
                self.masks_dir / f"{img_p.stem}_mask.png",
                self.masks_dir / f"{img_p.name}",
            ]
            for mc in mask_candidates:
                if mc.exists():
                    self.pairs.append((img_p, mc))
                    break
                    
        if len(self.pairs) == 0:
            print(f"Warning: No pairs found in {images_dir} and {masks_dir}")
            
        # ImageNet normalization statistics
        self.mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        self.std = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]
        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        # Synchronized resize
        image = image.resize(self.img_size, Image.BILINEAR)
        mask = mask.resize(self.img_size, Image.NEAREST)

        # Spatial augmentations during training
        if self.augment:
            if random.random() > 0.5:
                image = image.transpose(Image.FLIP_LEFT_RIGHT)
                mask = mask.transpose(Image.FLIP_LEFT_RIGHT)
            if random.random() > 0.5:
                image = image.transpose(Image.FLIP_TOP_BOTTOM)
                mask = mask.transpose(Image.FLIP_TOP_BOTTOM)

        # Convert image to Tensor & normalize
        img_np = np.array(image, dtype=np.float32) / 255.0
        img_norm = (img_np - self.mean) / self.std
        img_tensor = torch.from_numpy(img_norm).permute(2, 0, 1).float()

        # Convert mask to binary Tensor (0 or 1)
        mask_np = np.array(mask, dtype=np.float32)
        mask_binary = (mask_np > 127).astype(np.float32)
        mask_tensor = torch.from_numpy(mask_binary).unsqueeze(0).float()

        return img_tensor, mask_tensor

# Create Datasets and DataLoaders
train_ds = PolypDataset(f"{PROCESSED_DATA_DIR}/images/train", f"{PROCESSED_DATA_DIR}/masks/train", augment=True)
val_ds = PolypDataset(f"{PROCESSED_DATA_DIR}/images/val", f"{PROCESSED_DATA_DIR}/masks/val", augment=False)
test_ds = PolypDataset(f"{PROCESSED_DATA_DIR}/images/test", f"{PROCESSED_DATA_DIR}/masks/test", augment=False)

BATCH_SIZE = 8
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"DataLoaders ready: Train={len(train_ds)}, Val={len(val_ds)}, Test={len(test_ds)}")


### Step 7: ResUNet++ Architecture Implementation
This reproduces the exact architecture by **Jha et al. (2019)**:
- **Stem Block**: Initial residual conv feature extraction
- **Residual Convolution Units**: Identity shortcuts with BatchNorm and ReLU
- **Squeeze-and-Excitation (SE) Units**: Adaptive channel-wise feature recalibration
- **ASPP (Atrous Spatial Pyramid Pooling)**: Multi-scale receptive fields with dilation rates `[1, 6, 12, 18]`
- **Attention Gate (AG)**: Skip-connection filtering to suppress irrelevant background colonoscopy textures

In [ ]:
class SqueezeAndExcitation(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        reduced = max(channels // reduction, 4)
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(channels, reduced, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(reduced, channels, bias=False),
            nn.Sigmoid(),
        )
    def forward(self, x):
        b, c, _, _ = x.shape
        w = self.fc(x).view(b, c, 1, 1)
        return x * w

class ResidualConvSE(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv_block = nn.Sequential(
            nn.BatchNorm2d(in_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
        )
        self.se = SqueezeAndExcitation(out_ch)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )
    def forward(self, x):
        return self.shortcut(x) + self.se(self.conv_block(x))

class ASPP(nn.Module):
    def __init__(self, in_ch, out_ch, rates=(1, 6, 12, 18)):
        super().__init__()
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=1 if r == 1 else 3, padding=0 if r == 1 else r, dilation=r, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
            ) for r in rates
        ])
        self.project = nn.Sequential(
            nn.Conv2d(out_ch * len(rates), out_ch, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        outs = [b(x) for b in self.branches]
        return self.project(torch.cat(outs, dim=1))

class AttentionGate(nn.Module):
    def __init__(self, g_ch, x_ch, inter_ch):
        super().__init__()
        self.W_g = nn.Sequential(nn.Conv2d(g_ch, inter_ch, kernel_size=1, bias=False), nn.BatchNorm2d(inter_ch))
        self.W_x = nn.Sequential(nn.Conv2d(x_ch, inter_ch, kernel_size=1, bias=False), nn.BatchNorm2d(inter_ch))
        self.psi = nn.Sequential(nn.Conv2d(inter_ch, 1, kernel_size=1, bias=False), nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)
    def forward(self, g, x):
        psi = self.relu(self.W_g(g) + self.W_x(x))
        return x * self.psi(psi)

class ResUNetPlusPlus(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, filters=[32, 64, 128, 256, 512]):
        super().__init__()
        f = filters
        # Stem
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, f[0], kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(f[0]), nn.ReLU(inplace=True),
            nn.Conv2d(f[0], f[0], kernel_size=3, padding=1, bias=False),
        )
        self.stem_sc = nn.Conv2d(in_channels, f[0], kernel_size=1, bias=False)
        # Encoders
        self.enc1 = ResidualConvSE(f[0], f[1], stride=2)
        self.enc2 = ResidualConvSE(f[1], f[2], stride=2)
        self.enc3 = ResidualConvSE(f[2], f[3], stride=2)
        # ASPP Bridge
        self.bridge_aspp = ASPP(f[3], f[4])
        # Decoder 3
        self.up3 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.ag3 = AttentionGate(g_ch=f[4], x_ch=f[2], inter_ch=f[2] // 2)
        self.dec3 = ResidualConvSE(f[4] + f[2], f[3])
        # Decoder 2
        self.up2 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.ag2 = AttentionGate(g_ch=f[3], x_ch=f[1], inter_ch=f[1] // 2)
        self.dec2 = ResidualConvSE(f[3] + f[1], f[2])
        # Decoder 1
        self.up1 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.ag1 = AttentionGate(g_ch=f[2], x_ch=f[0], inter_ch=f[0] // 2)
        self.dec1 = ResidualConvSE(f[2] + f[0], f[1])
        # Output ASPP & Head
        self.out_aspp = ASPP(f[1], f[0])
        self.head = nn.Conv2d(f[0], out_channels, kernel_size=1)

    def forward(self, x):
        x0 = self.stem(x) + self.stem_sc(x)  # [B, 32, H, W]
        x1 = self.enc1(x0)                   # [B, 64, H/2, W/2]
        x2 = self.enc2(x1)                   # [B, 128, H/4, W/4]
        x3 = self.enc3(x2)                   # [B, 256, H/8, W/8]
        
        b = self.bridge_aspp(x3)             # [B, 512, H/8, W/8]
        
        u3 = self.up3(b)                     # [B, 512, H/4, W/4]
        s2 = self.ag3(g=u3, x=x2)            # [B, 128, H/4, W/4]
        d3 = self.dec3(torch.cat([u3, s2], dim=1)) # [B, 256, H/4, W/4]
        
        u2 = self.up2(d3)                    # [B, 256, H/2, W/2]
        s1 = self.ag2(g=u2, x=x1)            # [B, 64, H/2, W/2]
        d2 = self.dec2(torch.cat([u2, s1], dim=1)) # [B, 128, H/2, W/2]
        
        u1 = self.up1(d2)                    # [B, 128, H, W]
        s0 = self.ag1(g=u1, x=x0)            # [B, 32, H, W]
        d1 = self.dec1(torch.cat([u1, s0], dim=1)) # [B, 64, H, W]
        
        oa = self.out_aspp(d1)               # [B, 32, H, W]
        out = torch.sigmoid(self.head(oa))   # [B, 1, H, W]
        return out

model = ResUNetPlusPlus().to(device)
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ ResUNet++ Model instantiated on {device}.")
print(f"   Trainable Parameters: {num_params:,} (~{num_params / 1e6:.2f}M)")

# Test forward pass
dummy = torch.randn(2, 3, 256, 256).to(device)
with torch.no_grad():
    pred = model(dummy)
print(f"   Input Shape: {dummy.shape} -> Output Shape: {pred.shape} (Range: [{pred.min():.2f}, {pred.max():.2f}])")


### Step 8: Loss Function & Evaluation Metrics
We use a **Hybrid BCE + Dice Loss**:
$$\mathcal{L}_{total} = 0.5 \cdot \mathcal{L}_{BCE} + 0.5 \cdot \mathcal{L}_{Dice}$$

This handles both pixel-level cross-entropy and area-level overlap balance.

In [ ]:
class BCEDiceLoss(nn.Module):
    def __init__(self, alpha=0.5, beta=0.5, smooth=1e-6):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.smooth = smooth
        self.bce = nn.BCELoss()

    def forward(self, pred, target):
        bce_loss = self.bce(pred, target)
        p_flat = pred.contiguous().view(-1)
        t_flat = target.contiguous().view(-1)
        intersection = (p_flat * t_flat).sum()
        dice_loss = 1.0 - ((2.0 * intersection + self.smooth) / (p_flat.sum() + t_flat.sum() + self.smooth))
        return self.alpha * bce_loss + self.beta * dice_loss

def calculate_metrics(pred_prob, target, threshold=0.5, smooth=1e-6):
    """Computes Dice, IoU, Precision, Recall, Specificity."""
    p_bin = (pred_prob >= threshold).float()
    t_bin = (target >= 0.5).float()
    
    p_flat = p_bin.view(-1)
    t_flat = t_bin.view(-1)
    
    tp = (p_flat * t_flat).sum().item()
    fp = (p_flat * (1.0 - t_flat)).sum().item()
    fn = ((1.0 - p_flat) * t_flat).sum().item()
    tn = ((1.0 - p_flat) * (1.0 - t_flat)).sum().item()
    
    dice = (2.0 * tp + smooth) / (2.0 * tp + fp + fn + smooth)
    iou = (tp + smooth) / (tp + fp + fn + smooth)
    precision = (tp + smooth) / (tp + fp + smooth)
    recall = (tp + smooth) / (tp + fn + smooth)
    specificity = (tn + smooth) / (tn + fp + smooth)
    
    return {
        "dice": dice,
        "iou": iou,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
    }

criterion = BCEDiceLoss()
print("Loss and metric calculators configured.")


### Step 9: Training Loop with Validation & Early Stopping

In [ ]:
import time

EPOCHS = 50
LEARNING_RATE = 1e-4
PATIENCE = 12
CHECKPOINT_PATH = "resunetpp_best.pth"

optimizer = Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=5, min_lr=1e-6)

history = {
    "train_loss": [], "val_loss": [],
    "val_dice": [], "val_iou": [],
}

best_val_dice = 0.0
patience_counter = 0

print(f"Beginning training for {EPOCHS} epochs...")
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    # --- Training Phase ---
    model.train()
    train_loss_total = 0.0
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        preds = model(imgs)
        loss = criterion(preds, masks)
        loss.backward()
        optimizer.step()
        train_loss_total += loss.item() * imgs.size(0)
        
    train_loss = train_loss_total / max(len(train_ds), 1)
    
    # --- Validation Phase ---
    model.eval()
    val_loss_total = 0.0
    all_val_preds = []
    all_val_targets = []
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model(imgs)
            loss = criterion(preds, masks)
            val_loss_total += loss.item() * imgs.size(0)
            all_val_preds.append(preds.cpu())
            all_val_targets.append(masks.cpu())
            
    val_loss = val_loss_total / max(len(val_ds), 1)
    val_cat_preds = torch.cat(all_val_preds, dim=0) if all_val_preds else torch.empty(0)
    val_cat_targets = torch.cat(all_val_targets, dim=0) if all_val_targets else torch.empty(0)
    
    if len(val_cat_preds) > 0:
        val_metrics = calculate_metrics(val_cat_preds, val_cat_targets)
    else:
        val_metrics = {"dice": 0.0, "iou": 0.0}
        
    val_dice = val_metrics["dice"]
    val_iou = val_metrics["iou"]
    
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_dice"].append(val_dice)
    history["val_iou"].append(val_iou)
    
    scheduler.step(val_dice)
    current_lr = optimizer.param_groups[0]["lr"]
    
    # Checkpoint saving on best Val Dice
    is_best = val_dice > best_val_dice
    if is_best:
        best_val_dice = val_dice
        patience_counter = 0
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "val_dice": val_dice,
            "val_iou": val_iou,
            "config": {"model": "resunetpp", "filters": [32, 64, 128, 256, 512]},
        }, CHECKPOINT_PATH)
        best_marker = "⭐ (BEST)"
    else:
        patience_counter += 1
        best_marker = ""
        
    print(f"Epoch [{epoch:03d}/{EPOCHS:03d}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Dice: {val_dice:.4f} | Val IoU: {val_iou:.4f} | LR: {current_lr:.1e} {best_marker}")
    
    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping triggered at epoch {epoch} (no improvement in {PATIENCE} epochs).")
        break

elapsed = (time.time() - start_time) / 60
print(f"\nTraining finished in {elapsed:.1f} minutes. Best Val Dice: {best_val_dice:.4f}")


### Step 10: Training Convergence Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(history["train_loss"], label="Train Loss", color="#1976d2", lw=2)
axes[0].plot(history["val_loss"], label="Val Loss", color="#d32f2f", lw=2, linestyle="--")
axes[0].set_title("BCE + Dice Loss over Epochs", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Metric curves
axes[1].plot(history["val_dice"], label="Val Dice Coefficient", color="#388e3c", lw=2)
axes[1].plot(history["val_iou"], label="Val mIoU (Jaccard)", color="#f57c00", lw=2, linestyle="--")
axes[1].set_title("Validation Metrics over Epochs", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Score")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.savefig("training_curves.png", dpi=300)
plt.show()


### Step 11: Scientific Benchmarking on Unseen Test Set
Load the best checkpoint and evaluate across all key medical segmentation metrics.

In [ ]:
# Load best weights
ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"Loaded best model from epoch {ckpt['epoch']} (Val Dice: {ckpt['val_dice']:.4f})")

test_preds, test_targets = [], []
with torch.no_grad():
    for imgs, masks in test_loader:
        imgs = imgs.to(device)
        preds = model(imgs)
        test_preds.append(preds.cpu())
        test_targets.append(masks.cpu())

cat_preds = torch.cat(test_preds, dim=0)
cat_targets = torch.cat(test_targets, dim=0)
metrics = calculate_metrics(cat_preds, cat_targets)

print("=" * 60)
print("           RESEUNET++ TEST SET BENCHMARK RESULTS           ")
print("=" * 60)
print(f"  Dice Similarity Coefficient (F1 Score) : {metrics['dice'] * 100:6.2f}%")
print(f"  Mean Intersection over Union (mIoU)    : {metrics['iou'] * 100:6.2f}%")
print(f"  Precision (Positive Predictive Value)  : {metrics['precision'] * 100:6.2f}%")
print(f"  Recall / Sensitivity                   : {metrics['recall'] * 100:6.2f}%")
print(f"  Specificity (True Negative Rate)       : {metrics['specificity'] * 100:6.2f}%")
print("=" * 60)


### Step 12: Qualitative Visualization (Predictions vs Ground Truth)
Visual inspection of model segmentations with original colonoscopy images and ground truth.

In [ ]:
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

model.eval()
sample_indices = min(4, len(test_ds))
fig, axes = plt.subplots(sample_indices, 4, figsize=(16, sample_indices * 4))

with torch.no_grad():
    for i in range(sample_indices):
        img_tensor, mask_tensor = test_ds[i]
        input_batch = img_tensor.unsqueeze(0).to(device)
        pred_prob = model(input_batch).squeeze().cpu().numpy()
        pred_mask = (pred_prob >= 0.5).astype(np.uint8)
        
        # Denormalize image for display
        img_np = img_tensor.permute(1, 2, 0).numpy()
        img_vis = np.clip(img_np * std + mean, 0.0, 1.0)
        gt_mask = mask_tensor.squeeze().numpy()
        
        # Overlay (red translucent mask)
        overlay = img_vis.copy()
        overlay[pred_mask == 1] = overlay[pred_mask == 1] * 0.5 + np.array([0.9, 0.1, 0.1]) * 0.5
        
        ax_row = axes[i] if sample_indices > 1 else axes
        ax_row[0].imshow(img_vis)
        ax_row[0].set_title("Input Colonoscopy Image")
        ax_row[0].axis("off")
        
        ax_row[1].imshow(gt_mask, cmap="gray")
        ax_row[1].set_title("Ground Truth Mask")
        ax_row[1].axis("off")
        
        ax_row[2].imshow(pred_prob, cmap="jet")
        ax_row[2].set_title("Prediction Probability Map")
        ax_row[2].axis("off")
        
        ax_row[3].imshow(overlay)
        ax_row[3].set_title("Segmented Overlay")
        ax_row[3].axis("off")

plt.tight_layout()
plt.savefig("qualitative_results.png", dpi=300)
plt.show()


### Step 13: Export Checkpoint for Local Streamlit Application
Download `resunetpp_best.pth`. Once downloaded to your computer, move it into your project folder:
`artifacts/checkpoints/resunetpp_best.pth`
Then start your Streamlit application with `streamlit run app.py`!

In [ ]:
try:
    from google.colab import files
    print("Triggering browser download for best model checkpoint...")
    files.download(CHECKPOINT_PATH)
    files.download("training_curves.png")
    files.download("qualitative_results.png")
    print("✅ Download started!")
except ImportError:
    print(f"Local environment detected. Checkpoint saved to: {os.path.abspath(CHECKPOINT_PATH)}")
